Add *No Distortion* class to PatternReframe dataset splits

In [6]:
# Imports and config ===
from pathlib import Path
import pandas as pd
import numpy as np

SEED = 42
PROCESSED_DIR = Path("processed")
BAHASA_NONE_CSV = Path("bahasa_nondistorted.csv")

# Variant choice:
#   "v1_balanced" — None ≈ one positive class (~192 in train). Primary.
#   "v2_maddela"  — Maddela 1:1 binary, scaled to available Bahasa pool.
VARIANT = "v1_balanced"

# Length filter for Bahasa pool, in CHARACTERS (matching PatternReframe's thought_length unit).
# v1 spans roughly [p1, p99] of PatternReframe's character distribution.
# v2 widens slightly to gain pool for the Maddela-scaled variant.
LEN_BOUNDS = {"v1_balanced": (40, 150), "v2_maddela": (30, 220)}[VARIANT]

# Per-split None counts.
# v1: matched to one PatternReframe positive class (~192/96/681).
# v2: scaled-down Maddela 1:1 binary; numbers computed below from pool size.
NONE_COUNTS_V1 = {"train": 192, "val": 96, "test": 681}

In [7]:
# Load the existing processed PatternReframe splits ===
pr_train = pd.read_csv(PROCESSED_DIR / "patternreframe_train.csv")
pr_val   = pd.read_csv(PROCESSED_DIR / "patternreframe_val.csv")
pr_test  = pd.read_csv(PROCESSED_DIR / "patternreframe_test.csv")

# Schema check — every None row written below must conform to this exact column order.
EXPECTED_COLS = [
    "persona", "pattern", "pattern_def", "thought", "thought_length",
    "n_reframes", "reframe_1", "reframe_2", "reframe_3",
    "marked_catastrophizing", "marked_discounting_positive", "marked_labeling",
    "marked_mental_filtering", "marked_mind_reading", "marked_fortune_telling",
    "marked_overgeneralization", "marked_personalization", "marked_black_and_white",
    "marked_should_statements", "marked_emotional_reasoning", "marked_none",
    "strategies_per_reframe",
]
for name, df in [("train", pr_train), ("val", pr_val), ("test", pr_test)]:
    missing = set(EXPECTED_COLS) - set(df.columns)
    extra   = set(df.columns) - set(EXPECTED_COLS)
    assert not missing, f"{name} missing columns: {missing}"
    if extra:
        print(f"{name}: extra columns present (will be carried through): {extra}")
    print(f"{name}: {len(df)} positives, columns OK")

train: 1920 positives, columns OK
val: 961 positives, columns OK
test: 6807 positives, columns OK


In [8]:
# Load and filter the Bahasa No Distortion pool ===
bah = pd.read_csv(PROCESSED_DIR / BAHASA_NONE_CSV)
print(f"Bahasa No Distortion CSV: {len(bah)} rows")
print(f"Length stats (characters): median={bah['text_english_length'].median():.0f}, "
      f"p25={bah['text_english_length'].quantile(0.25):.0f}, "
      f"p75={bah['text_english_length'].quantile(0.75):.0f}, "
      f"max={bah['text_english_length'].max()}")

lo, hi = LEN_BOUNDS
pool = bah[(bah["text_english_length"] >= lo) & (bah["text_english_length"] <= hi)].copy()
pool["text_english"] = pool["text_english"].astype(str).str.strip()
pool = pool.drop_duplicates(subset=["text_english"]).reset_index(drop=True)

# Deterministic shuffle.
pool = pool.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print(f"Pool after [{lo},{hi}]-word filter and dedup: {len(pool)}")

Bahasa No Distortion CSV: 2246 rows
Length stats (characters): median=120, p25=75, p75=184, max=2120
Pool after [40,150]-word filter and dedup: 1285


In [9]:
# Decide split sizes for the chosen variant ===
if VARIANT == "v1_balanced":
    n_train = NONE_COUNTS_V1["train"]
    n_val   = NONE_COUNTS_V1["val"]
    n_test  = NONE_COUNTS_V1["test"]
elif VARIANT == "v2_maddela":
    # Maddela target for full 1:1 binary at unscaled pool: 1914 / 954 / 6765 (sum 9633).
    target = (1914, 954, 6765)
    needed = sum(target)
    scale = len(pool) / needed
    n_train = int(round(target[0] * scale))
    n_val   = int(round(target[1] * scale))
    n_test  = len(pool) - n_train - n_val
    print(f"v2 scale factor: {scale:.3f}")
else:
    raise ValueError(VARIANT)

assert n_train + n_val + n_test <= len(pool), \
    f"Pool too small: need {n_train + n_val + n_test}, have {len(pool)}."
print(f"None split sizes: train={n_train}, val={n_val}, test={n_test}")

# Slice the pool into three non-overlapping chunks.
none_train_src = pool.iloc[:n_train].copy()
none_val_src   = pool.iloc[n_train:n_train + n_val].copy()
none_test_src  = pool.iloc[n_train + n_val:n_train + n_val + n_test].copy()

# Sanity: no shared text across splits.
assert not (set(none_train_src["text_english"]) & set(none_val_src["text_english"]))
assert not (set(none_train_src["text_english"]) & set(none_test_src["text_english"]))
assert not (set(none_val_src["text_english"])   & set(none_test_src["text_english"]))

None split sizes: train=192, val=96, test=681


In [10]:
# Build None rows that conform to the PatternReframe schema ===
PATTERN_DEF_NONE = (
    "No distortion: the thought does not exhibit any of the ten unhelpful "
    "thinking patterns covered by the dataset."
)

MARKED_COLS = [
    "marked_catastrophizing", "marked_discounting_positive", "marked_labeling",
    "marked_mental_filtering", "marked_mind_reading", "marked_fortune_telling",
    "marked_overgeneralization", "marked_personalization", "marked_black_and_white",
    "marked_should_statements", "marked_emotional_reasoning",
]

def build_none_rows(src: pd.DataFrame) -> pd.DataFrame:
    """Map a slice of the Bahasa pool into the full PatternReframe schema."""
    n = len(src)
    out = pd.DataFrame({
        "persona":                 [""] * n,                          # no persona for Bahasa
        "pattern":                 ["None"] * n,
        "pattern_def":             [PATTERN_DEF_NONE] * n,
        "thought":                 src["text_english"].values,
        "thought_length":          src["text_english_length"].values,
        "n_reframes":              0,
        "reframe_1":               np.nan,
        "reframe_2":               np.nan,
        "reframe_3":               np.nan,
        "marked_none":             1,                                  # full vote on "None"
        "strategies_per_reframe":  np.nan,
    })
    # All marked_<distortion> columns are zero for None records.
    for col in MARKED_COLS:
        out[col] = 0
    # Reorder columns to match the existing files exactly.
    out = out[EXPECTED_COLS]
    return out

none_train = build_none_rows(none_train_src)
none_val   = build_none_rows(none_val_src)
none_test  = build_none_rows(none_test_src)

# Schema parity check.
for name, df in [("train", none_train), ("val", none_val), ("test", none_test)]:
    assert list(df.columns) == EXPECTED_COLS, f"{name} column order mismatch"
    assert df["pattern"].eq("None").all()
print("None rows built and schema-aligned.")

None rows built and schema-aligned.


In [11]:
# Concatenate, shuffle, and write augmented splits ===
def merge_and_shuffle(positives: pd.DataFrame, none_df: pd.DataFrame) -> pd.DataFrame:
    merged = pd.concat([positives, none_df], ignore_index=True)
    return merged.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

aug_train = merge_and_shuffle(pr_train, none_train)
aug_val   = merge_and_shuffle(pr_val,   none_val)
aug_test  = merge_and_shuffle(pr_test,  none_test)

suffix = VARIANT  # e.g. "v1_balanced" or "v2_maddela"
aug_train.to_csv(PROCESSED_DIR / f"patternreframe_train_aug_{suffix}.csv", index=False)
aug_val.to_csv(  PROCESSED_DIR / f"patternreframe_val_aug_{suffix}.csv",   index=False)
aug_test.to_csv( PROCESSED_DIR / f"patternreframe_test_aug_{suffix}.csv",  index=False)

print(f"\nAugmented split sizes ({suffix}):")
print(f"  train: {len(aug_train)}  ({len(pr_train)} positives + {len(none_train)} None)")
print(f"  val:   {len(aug_val)}    ({len(pr_val)} positives + {len(none_val)} None)")
print(f"  test:  {len(aug_test)}   ({len(pr_test)} positives + {len(none_test)} None)")

print("\nClass distribution in augmented train:")
print(aug_train["pattern"].value_counts().to_string())


Augmented split sizes (v1_balanced):
  train: 2112  (1920 positives + 192 None)
  val:   1057    (961 positives + 96 None)
  test:  7488   (6807 positives + 681 None)

Class distribution in augmented train:
pattern
Catastrophizing                                                    200
Jumping to conclusions: Fortune-telling                            199
Jumping to conclusions: mind reading                               197
Overgeneralization                                                 197
Labeling and mislabeling                                           196
Discounting the positive                                           194
None                                                               192
Black-and-white or polarized thinking / All or nothing thinking    188
Personalization                                                    185
Mental filtering                                                   185
Should statements                                                  179


In [15]:
# sanity check that None rows look right ===
print("Random sample of 5 None records from the augmented train set:\n")
none_in_train = aug_train[aug_train["pattern"] == "None"]
for _, row in none_in_train.sample(5).iterrows():
    print(f"  ({row['thought_length']} chars) {row['thought']}")
    print(f"    marked_none={row['marked_none']}, n_reframes={row['n_reframes']}, "
          f"persona={'<empty>' if not row['persona'] else row['persona'][:40]+'...'}")
    print()

Random sample of 5 None records from the augmented train set:

  (88 chars) I'm thinking about what I want to do in the future, my way of studying is not consistent
    marked_none=1, n_reframes=0, persona=<empty>

  (105 chars) Remain yourself and always be satisfied with your own achievements without comparing yourself with others
    marked_none=1, n_reframes=0, persona=<empty>

  (62 chars) when you meet a friend somewhere but he doesn't respond to her
    marked_none=1, n_reframes=0, persona=<empty>

  (61 chars) When my father went to prison for gambling he sold the house.
    marked_none=1, n_reframes=0, persona=<empty>

  (133 chars) I always wanted peace and tranquility in my romantic relationships because of the traumatic experiences of my parents' relationships.
    marked_none=1, n_reframes=0, persona=<empty>



The split sizes are exactly what was targeted, and the augmented train is now an 11-class problem with all classes within a tight 179–200 band. The None records read naturally as self-reflective writing about personal situations, these will not be trivially separable from PatternReframe positives, which is what we want.

PatternReframe thought_length is characters -> filter applied at [40, 150], the resulting per-class balance (179–200 in train), and the deliberate departure from Maddela's 10x None ratio with v2 retained as an appendix comparison (todo).